In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Working dir:", os.getcwd())

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

# Adjust path below to where your step7_pca.csv is located in your Drive
RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed/Dataset7.csv"
if not os.path.exists(RAW_PATH):
    # fallback: try current folder
    RAW_PATH = "Dataset7.csv"

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, "shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


In [ ]:
# Cell 2: Prepare features and target (shared)
from sklearn.model_selection import train_test_split

# Ensure target exists
assert 'selling_price' in df.columns, "selling_price not in dataset"

X = df.drop(columns=['selling_price'])
y = df['selling_price']

# If any NaNs remain, fill with median for modeling convenience
X = X.fillna(X.median())

# Train-test split (same for all members to ensure fair comparison)
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error

depths = [3, 5, 8, 12]
results_dt = []
for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_SEED)
    dt.fit(X_train, y_train)
    yp = dt.predict(X_test)
    results_dt.append((d, r2_score(y_test, yp), mean_absolute_error(y_test, yp)))

# Print results
for d, r2v, mae in results_dt:
    print(f"Depth {d} -> R2: {r2v:.3f}, MAE: {mae:.0f}")

# Visualization: Depth vs R2
depths_list = [r[0] for r in results_dt]
r2_list = [r[1] for r in results_dt]
plt.figure(figsize=(6,4))
plt.plot(depths_list, r2_list, marker='o')
plt.xlabel("max_depth")
plt.ylabel("R² Score")
plt.title("Decision Tree: max_depth vs R²")
plt.grid(True)
plt.show()

# Choose best depth by R2 and show actual vs predicted
best_depth = max(results_dt, key=lambda t: t[1])[0]
best_dt = DecisionTreeRegressor(max_depth=best_depth, random_state=RANDOM_SEED)
best_dt.fit(X_train, y_train)
y_pred_dt = best_dt.predict(X_test)

plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_dt, alpha=0.5, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"Decision Tree (depth={best_depth}) Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

print("Decision Trees capture nonlinear relationships; increasing depth improves fit up to a point but risks overfitting.")